In [0]:
storage_account = "stbankamldev"

storage_key = dbutils.secrets.get(scope = "aml-scope" , key = "storage-access-key")
spark.conf.set("fs.azure.account.key.stbankamldev.blob.core.windows.net", storage_key)
df_realtionship_raw = spark.read.option("header","true").option("inferSchema","true").csv("wasbs://raw@stbankamldev.blob.core.windows.net/counterparty_relationship_history.csv")



df_realtionship_raw.printSchema()
df_realtionship_raw.show(10)


In [0]:
from pyspark.sql.functions import *

df_relationship_clean = df_realtionship_raw \
.withColumn("clean_relationship_status", upper(trim(col("relationship_status")))) \
.withColumn("clean_counterparty_country", upper(trim(col("counterparty_country")))) 


display(df_relationship_clean)



In [0]:
df_transaction_alerts = spark.read.parquet("wasbs://gold@stbankamldev.blob.core.windows.net/transaction_alerts")

df_transaction_alerts.show()

In [0]:
df_relationship_clean.groupBy(col("customer_id"),col("counterparty_account_id")).count().orderBy(col("count").desc()).show()

In [0]:
df_transaction_relationship = df_transaction_alerts.alias("t").join(df_relationship_clean.alias("r"),(col("t.customer_id") == col("r.customer_id")) & (col("r.counterparty_account_id") == col("t.counterparty_account_id")) , "left"  ) \
    .select(col("t.transaction_id"),col("t.customer_id"),col("t.counterparty_account_id"),col("t.clean_counterparty_country"),
            col("r.clean_relationship_status"),col("r.first_seen_date"),col("t.clean_transaction_amount"),col("t.transaction_risk_score"))

df_transaction_relationship.printSchema()
df_transaction_relationship.show()
    #.withColumn("new_relationship_flag" , when(col("")))

In [0]:
from pyspark.sql.functions import concat_ws, col, when 
df_transaction_relationship_alert = df_transaction_relationship \
.withColumn("unknown_relationship_flag",when((col("clean_relationship_status").isNull()) & (col("first_seen_date").isNull()) , 1 ).otherwise(0)) \
.withColumn("new_relationship_flag",when(col("first_seen_date").isNull(),1).otherwise(0)) \
.withColumn("relationship_risk_score", col("unknown_relationship_flag")+col("new_relationship_flag")) \
.withColumn("relationship_risk_level", when(col("relationship_risk_score") >= 2 , "HIGH")
                                        .when(col("relationship_risk_score") >=1, "MEDIUM")
                                        .otherwise("LOW")
                                           )    \
.withColumn("relationship_risk_reason" , concat_ws("|",when(col("unknown_relationship_flag") == 1 , "UNKNOWN_RELATIONSHIP"),
                                         when(col("new_relationship_flag") == 1 , "NEW_COUNTERPARTY"))
                                              ) \
.select("transaction_id","customer_id","counterparty_account_id","clean_counterparty_country","clean_transaction_amount",
"transaction_risk_score","relationship_risk_score","relationship_risk_level","relationship_risk_reason")

#df_transaction_relationship_alert.printSchema()
#df_transaction_relationship_alert.groupBy(col("customer_id")).count().orderBy(col("count").desc()).show(20)
#df_transaction_relationship_alert.show()
storage_account = "stbankamldev"

storage_key = dbutils.secrets.get(scope = "aml-scope" , key = "storage-access-key")
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    storage_key
)

df_transaction_relationship_alert.write.mode("overwrite").parquet("wasbs://gold@stbankamldev.blob.core.windows.net/relationship_risk_alerts")
df_relationsh_gold = spark.read.parquet("wasbs://gold@stbankamldev.blob.core.windows.net/relationship_risk_alerts")
df_relationsh_gold.show()

In [0]:
df_relationsh_gold.groupBy("customer_id").count().filter(col("count") > 1).orderBy(col("count").desc()).show()
df_relationsh_gold.filter(col("customer_id") == "C005").show()
display(f"relationship risk alerts count: {df_relationsh_gold.count()}")